# Лекция 09. Стек и очередь

Стек и очередь — не сложные контейнеры, а два простых правила доступа. Стек сначала возвращает последнее добавленное, очередь — первое. Из этих правил вырастают история отмены, проверка вложенности, обслуживание заявок и алгоритмы скользящего окна.

## Цели

После лекции вы сможете:

- различать LIFO и FIFO;
- реализовывать стек через `list`, очередь через `deque`;
- объяснять стоимость операций и амортизированный анализ;
- применять стек к undo и вложенным скобкам;
- применять очередь к заявкам и циклической обработке;
- поддерживать последние значения ограниченным `deque`;
- вычислять среднее и максимум скользящего окна за линейное время;
- проверять инварианты этих структур.

## Перед началом

Нужны списки, циклы, функции, оценки сложности и итераторы. На стек и LIFO заложено около 25 минут, на очередь, `deque` и амортизированную стоимость — 25 минут, на прикладные алгоритмы — 25 минут, на контринтуитивные примеры, самопроверку и вопросы — 15 минут.

## Сначала правило, потом контейнер

**Абстрактный тип данных** описывает допустимые операции и их смысл, но не конкретное хранение.

- Стек: добавить на вершину, посмотреть вершину, снять вершину. Правило **LIFO** — last in, first out.
- Очередь: добавить в хвост, посмотреть голову, забрать голову. Правило **FIFO** — first in, first out.

Один и тот же контейнер иногда умеет больше операций, но прикладной код должен соблюдать выбранный контракт.

## Стек: последнее изменение отменяется первым

Редактор хранит снимки или обратные команды. После трёх изменений команда undo должна снять третье, затем второе. Это ровно LIFO.

In [ ]:
history = []
history.append("added income")
history.append("changed tax")
history.append("deleted rent")

last_change = history.pop()
assert last_change == "deleted rent"
assert history == ["added income", "changed tax"]

## Стек через `list`

Правый конец списка подходит для стека:

| Операция стека | Python | Ожидаемая стоимость |
|---|---|---:|
| push | `stack.append(item)` | амортизированная `O(1)` |
| peek | `stack[-1]` | `O(1)` |
| pop | `stack.pop()` | `O(1)` |
| is empty | `not stack` | `O(1)` |

Если стек пуст, `stack[-1]` и `stack.pop()` поднимают `IndexError`. Контракт функции должен заранее определить, является это ошибкой или нормальной ситуацией.

## История команд без классов

До занятия 10 достаточно обычной функции и списка. Для `undo` проверяем пустоту перед `pop`, чтобы пустая отмена имела явно выбранное поведение.

In [ ]:
def apply_history(commands):
    stack = []
    for action, value in commands:
        if action == "add":
            stack.append(value)
        elif action == "undo" and stack:
            stack.pop()
    return stack

commands = [("add", "income"), ("add", "tax"), ("undo", None)]
assert apply_history(commands) == ["income"]

## Вложенные скобки: незакрытая открывающая ждёт в стеке

При чтении слева направо:

1. открывающую скобку кладём в стек;
2. закрывающая должна соответствовать вершине;
3. после корректной пары снимаем вершину;
4. в конце стек должен быть пуст.

Почему стек? Первая открытая скобка закрывается последней, а последняя открытая — первой.

In [ ]:
def brackets_are_valid(text):
    expected_opening = {")": "(", "]": "[", "}": "{"}
    openings = set(expected_opening.values())
    stack = []

    for symbol in text:
        if symbol in openings:
            stack.append(symbol)
        elif symbol in expected_opening:
            if not stack or stack.pop() != expected_opening[symbol]:
                return False

    return not stack

assert brackets_are_valid("revenue - (tax + fee[0])")
assert not brackets_are_valid("([)]")
assert not brackets_are_valid("amount)")

### Инвариант проверки

После обработки любого префикса текста стек содержит ровно те открывающие скобки, которым ещё не нашлась пара, в порядке их появления. Если очередная закрывающая не совпадает с вершиной, никакое продолжение текста уже не исправит порядок — можно завершить сразу.

## Очередь: первая заявка обслуживается первой

Если нет приоритетов и специальных правил, заявка, пришедшая раньше, должна быть обработана раньше. Добавляем справа, забираем слева — FIFO.

In [ ]:
from collections import deque

requests = deque()
requests.append({"id": "r-1", "topic": "refund"})
requests.append({"id": "r-2", "topic": "delivery"})

first_request = requests.popleft()
assert first_request["id"] == "r-1"
assert requests[0]["id"] == "r-2"

## Почему не `list.pop(0)`

Список хранит элементы в непрерывном массиве ссылок. После удаления позиции 0 остальные ссылки надо сдвинуть влево — это `O(N)`. Если повторить для всех элементов, обслуживание очереди может стать квадратичным.

`deque` спроектирован для добавления и удаления с обоих концов с приблизительно одинаковой `O(1)` стоимостью. Для очереди используем `append` и `popleft`.

## Основные операции `deque`

| Действие | Справа | Слева |
|---|---|---|
| добавить | `append(x)` | `appendleft(x)` |
| удалить и вернуть | `pop()` | `popleft()` |
| посмотреть | `queue[-1]` | `queue[0]` |
| добавить несколько | `extend(xs)` | `extendleft(xs)` |

`deque` является двусторонней очередью. Но для обычной FIFO-очереди лучше придерживаться одной пары операций, чтобы направление читалось без мысленного переворота.

## Обработка событий очереди

События прихода и обслуживания можно воспроизвести одним проходом. Результат отдельно хранит порядок реально обслуженных заявок.

In [ ]:
def served_requests(events):
    waiting = deque()
    served = []
    for action, request_id in events:
        if action == "arrive":
            waiting.append(request_id)
        elif action == "serve" and waiting:
            served.append(waiting.popleft())
    return served

events = [("arrive", "r-1"), ("arrive", "r-2"), ("serve", None)]
assert served_requests(events) == ["r-1"]

## Амортизированная сложность

`list.append()` обычно записывает ссылку в свободную позицию — `O(1)`. Иногда массиву не хватает места: Python выделяет больший блок и переносит ссылки, что стоит `O(N)`.

Амортизированный анализ рассматривает длинную последовательность операций. Дорогие расширения происходят редко, поэтому суммарная стоимость `N` добавлений остаётся `O(N)`, а средняя стоимость одного — `O(1)`. Это не то же самое, что обещать `O(1)` для каждого отдельного `append`.

## Ограниченный `deque`: только последние события

`deque(maxlen=k)` автоматически хранит не более `k` элементов. При добавлении в полный контейнер элемент с противоположного конца удаляется. Это удобно для последних запросов, строк лога или значений метрики.

In [ ]:
recent_errors = deque(maxlen=3)
for error in ["timeout", "denied", "invalid", "unavailable"]:
    recent_errors.append(error)

assert list(recent_errors) == ["denied", "invalid", "unavailable"]

## Циклическая очередь задач

При round-robin каждая задача получает один шаг. Берём задачу слева; если она не завершена, возвращаем справа. Это обычная очередь, а не очередь с приоритетом.

In [ ]:
def completion_order(tasks):
    queue = deque((task["id"], task["steps"]) for task in tasks)
    completed = []

    while queue:
        task_id, steps = queue.popleft()
        steps -= 1
        if steps == 0:
            completed.append(task_id)
        else:
            queue.append((task_id, steps))

    return completed

tasks = [{"id": "a", "steps": 1}, {"id": "b", "steps": 3}, {"id": "c", "steps": 2}]
assert completion_order(tasks) == ["a", "c", "b"]

## Скользящее окно

Вместо пересчёта последних `k` значений с нуля поддерживаем только изменение окна:

- новое значение входит справа;
- старое выходит слева;
- сохранённое агрегированное состояние обновляется.

Для среднего достаточно очереди значений и текущей суммы.

In [ ]:
def moving_average(values, window):
    if not 1 <= window <= len(values):
        raise ValueError("invalid window")

    current = deque()
    current_sum = 0
    result = []

    for value in values:
        current.append(value)
        current_sum += value
        if len(current) > window:
            current_sum -= current.popleft()
        if len(current) == window:
            result.append(current_sum / window)

    return result

assert moving_average([10, 20, 30, 50], 3) == [20.0, 100 / 3]

### Почему среднее вычисляется за `O(N)`

Каждое значение ровно один раз добавляется и не более одного раза удаляется. Обновление суммы занимает `O(1)`, поэтому весь проход — `O(N)`. Память ограничена `O(window)`. Пересчёт `sum(values[i:i+k])` для каждого окна дал бы `O(Nk)` и создавал бы срезы.

## Монотонная очередь для максимума окна

Обычный `deque` хранит все значения окна. Для максимума можно хранить только **кандидатов** — их индексы в порядке невозрастания значений.

Перед добавлением индекса `i`:

1. удаляем слева индексы вне окна;
2. удаляем справа индексы со значением `<= values[i]`: новый элемент моложе и не меньше, старые больше не станут максимумом;
3. добавляем `i`;
4. максимум находится по индексу слева.

Каждый индекс входит и выходит не более одного раза, поэтому время `O(N)`.

In [ ]:
def sliding_maximum(values, window):
    if not 1 <= window <= len(values):
        raise ValueError("invalid window")

    candidates = deque()
    result = []

    for index, value in enumerate(values):
        while candidates and candidates[0] <= index - window:
            candidates.popleft()
        while candidates and values[candidates[-1]] <= value:
            candidates.pop()
        candidates.append(index)
        if index >= window - 1:
            result.append(values[candidates[0]])

    return result

assert sliding_maximum([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]

### Почему храним индексы

По одному значению нельзя понять, покинул ли конкретный экземпляр окно, особенно при повторах. Индекс одновременно даёт значение `values[index]` и возраст элемента. Левый кандидат является максимумом, пока его индекс больше `i - window`.

## `rotate`: сдвиг концов, а не сортировка

`queue.rotate(k)` переносит элементы между концами: положительное `k` двигает вправо, отрицательное — влево. Операция полезна для циклического просмотра, но не меняет относительный круговой порядок.

In [ ]:
agents = deque(["Анна", "Борис", "Вера"])
agents.rotate(-1)
assert list(agents) == ["Борис", "Вера", "Анна"]

## Границы применимости

- Стек не нужен, если требуется удалить произвольный элемент.
- FIFO-очередь не выражает приоритеты; очередь с приоритетом и `heapq` появятся в занятии 11.
- `deque` хорош у концов, но не заменяет список для частого доступа к середине.
- Скользящее окно предполагает, что данные приходят в определённом порядке.
- Названия `stack`, `queue`, `push`, `waiting` помогают не использовать случайно лишние операции контейнера.

## Что проверять

Для стека и очереди нужны не только красивые непустые примеры:

- пустой вход и одна запись;
- попытка снять элемент из пустого контейнера;
- несколько одинаковых значений;
- порядок LIFO или FIFO;
- окно длины 1 и длины всего списка;
- возрастающие и убывающие данные;
- неизменность исходных записей;
- инвариант стека скобок или монотонной очереди после каждого шага.

## Неожиданно, но по правилам

Сначала предскажите результат. Все эффекты следуют из направления операции или контракта ограниченного контейнера.

### 1. `pop()` одновременно изменяет список и возвращает элемент

Результат — снятая вершина, а не изменённый стек. Если присвоить `stack = stack.pop()`, имя перестанет указывать на список.

In [ ]:
stack = ["first", "second"]
removed = stack.pop()
assert removed == "second"
assert stack == ["first"]

### 2. Полный `deque(maxlen=...)` не поднимает ошибку при `append`

Он молча удаляет элемент с противоположного конца. Это удобно для последних событий, но опасно, если потеря данных не была частью контракта.

In [ ]:
recent = deque([1, 2, 3], maxlen=3)
recent.append(4)
assert list(recent) == [2, 3, 4]
recent.appendleft(1)
assert list(recent) == [1, 2, 3]

### 3. `extendleft` разворачивает порядок входа

Метод последовательно делает `appendleft` для каждого элемента. Следующий элемент каждый раз оказывается перед предыдущим.

In [ ]:
queue = deque([4])
queue.extendleft([1, 2, 3])
assert list(queue) == [3, 2, 1, 4]

### 4. Положительный `rotate` двигает вправо

Последний элемент становится первым. Для движения очереди к следующему исполнителю обычно нужен `rotate(-1)`.

In [ ]:
items = deque([1, 2, 3, 4])
items.rotate(1)
assert list(items) == [4, 1, 2, 3]

### 5. Индекс у `deque` существует, но середина не становится массивом

Доступ к обоим концам быстрый. Чтобы добраться до далёкого элемента в середине, реализации приходится пройти от ближайшего конца. Поэтому частый произвольный доступ — задача списка, а не очереди.

## Самопроверка

1. Чем LIFO отличается от FIFO?
2. Какие операции списка образуют стек?
3. Почему `list.pop(0)` имеет линейную стоимость?
4. Чем амортизированная `O(1)` отличается от гарантированной для каждого вызова?
5. Почему проверке скобок нужен стек?
6. Какие операции `deque` образуют FIFO-очередь?
7. Что происходит при переполнении `deque(maxlen=k)`?
8. Как обновлять среднее окна за `O(1)` на шаг?
9. Какой инвариант поддерживает монотонная очередь?
10. Почему она хранит индексы, а не только значения?

## Источники

- [`collections.deque`](https://docs.python.org/3/library/collections.html#collections.deque) — операции с обоими концами, `maxlen`, `rotate` и гарантии контейнера.
- [Using lists as stacks](https://docs.python.org/3/tutorial/datastructures.html#using-lists-as-stacks) — стек через `append` и `pop`.
- [Using lists as queues](https://docs.python.org/3/tutorial/datastructures.html#using-lists-as-queues) — почему для FIFO рекомендуется `deque`, а не удаление начала списка.

## Итоги

- Стек задаёт LIFO и естественно реализуется правым концом списка.
- Очередь задаёт FIFO; для неё подходят `deque.append` и `deque.popleft`.
- `list.append` имеет амортизированную `O(1)`, а `list.pop(0)` сдвигает элементы за `O(N)`.
- Стек хранит незавершённую вложенность и историю отмены.
- Очередь хранит порядок ожидания и поддерживает циклическую обработку.
- Ограниченный `deque` хранит последние значения.
- Скользящие агрегаты обновляются по входящему и уходящему элементу.
- Монотонная очередь оставляет только кандидатов на максимум и обрабатывает весь список за `O(N)`.